# Gemma 4 E2B Architecture Enumeration

Phase 1 Task B: Enumerate Gemma 4 E2B module structure to fill `notes/architecture.md`.

**Run on Colab T4** (16GB). E2B loads in ~10GB in bfloat16.

**Outputs needed:**
- Full module tree (`named_modules`)
- Config dump (attention pattern, num_kv_shared_layers, PLE config)
- HF class names
- PLE parameter shapes from state dict
- Forward pass trace to confirm PLE data flow

**HuggingFace access**: requires login with a token that has access to `google/gemma-4-E2B-it`.

In [ ]:
# Install dependencies
!pip install -q transformers accelerate

In [ ]:
from huggingface_hub import login
login()  # paste HF token with gemma-4 access

## 1. Config inspection (no weights loaded)

In [ ]:
from transformers import AutoConfig

MODEL_ID = "google/gemma-4-E2B-it"
cfg = AutoConfig.from_pretrained(MODEL_ID)

print("=== Config type ===")
print(type(cfg))
print()
print("=== Full config ===")
print(cfg)

In [ ]:
# Extract key fields — adjust attribute names if they differ
interesting = [
    'num_hidden_layers',
    'hidden_size',
    'num_attention_heads',
    'num_key_value_heads',
    'intermediate_size',
    'sliding_window',
    'attention_bias',
    'rope_theta',
    # PLE / shared KV — names TBD
    'num_kv_shared_layers',
    'use_per_layer_embeddings',
    'per_layer_embedding_size',
    # Attention type pattern
    'attention_types',
    'layer_types',
    'sliding_window_pattern',
    'attn_logit_softcapping',
    'query_pre_attn_scalar',
]

print("=== Key config fields ===")
for field in interesting:
    val = getattr(cfg, field, '(not found)')
    print(f"  {field}: {val}")

print()
print("=== All config keys ===")
print(sorted(cfg.to_dict().keys()))

## 2. Load model and enumerate modules

In [ ]:
import torch
from transformers import AutoModelForCausalLM

# Load in bfloat16 on T4 — uses ~10GB
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)
print(f"Loaded: {type(model).__name__}")
print(f"Config class: {type(model.config).__name__}")

In [ ]:
# Full module tree — paste output into notes/architecture.md
print("=== named_modules() ===")
for name, mod in model.named_modules():
    print(f"{name}: {type(mod).__name__}")

In [ ]:
# Per-layer breakdown: attention type and PLE presence
print("=== Per-layer module types ===")
# Access the decoder layers — adjust path if needed
layers = model.model.layers
print(f"Num layers: {len(layers)}")
print()

for i, layer in enumerate(layers):
    attn_type = type(layer.self_attn).__name__
    children = [n for n, _ in layer.named_children()]
    print(f"Layer {i:2d}: attn={attn_type}, children={children}")

In [ ]:
# PLE parameters in state dict
print("=== PLE-related parameters ===")
sd = model.state_dict()
ple_keys = [k for k in sd.keys() if 'per_layer' in k.lower() or 'ple' in k.lower()]
if ple_keys:
    for k in ple_keys:
        print(f"  {k}: {sd[k].shape}")
else:
    print("  (no keys matching 'per_layer' or 'ple' — check module tree for correct name)")

print()
print("=== Shared KV parameters ===")
kv_keys = [k for k in sd.keys() if 'shared' in k.lower() or 'kv_cache' in k.lower()]
if kv_keys:
    for k in kv_keys:
        print(f"  {k}: {sd[k].shape}")
else:
    print("  (no keys matching 'shared'/'kv_cache' — shared KV may be activation-based)")

In [ ]:
# Inspect layer 0 in detail
print("=== Layer 0 full detail ===")
print(layers[0])
print()
print("=== Layer 0 attention ===")
print(layers[0].self_attn)
print()
# Check if there's a sliding window attribute
attn = layers[0].self_attn
print("Attention attributes:", [a for a in dir(attn) if not a.startswith('_')])

In [ ]:
# Identify local vs global layers by inspecting sliding_window or is_sliding attributes
print("=== Per-layer attention type ===")
for i, layer in enumerate(layers):
    attn = layer.self_attn
    is_local = getattr(attn, 'is_sliding', None)
    sliding_window = getattr(attn, 'sliding_window', None)
    layer_type = getattr(attn, 'layer_type', None)
    print(f"Layer {i:2d}: is_sliding={is_local}, sliding_window={sliding_window}, layer_type={layer_type}")

## 3. Forward pass trace — confirm PLE data flow

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
text = "The capital of France is"
inputs = tokenizer(text, return_tensors='pt').to(model.device)

# Register forward hooks to trace what flows in/out of each layer
activation_shapes = {}

def make_hook(name):
    def hook(module, input, output):
        if isinstance(output, torch.Tensor):
            activation_shapes[name] = output.shape
        elif isinstance(output, (tuple, list)) and isinstance(output[0], torch.Tensor):
            activation_shapes[name] = output[0].shape
    return hook

handles = []
for i, layer in enumerate(layers[:4]):  # trace first 4 layers
    handles.append(layer.register_forward_hook(make_hook(f'layer_{i}')))
    handles.append(layer.self_attn.register_forward_hook(make_hook(f'layer_{i}_attn')))
    handles.append(layer.mlp.register_forward_hook(make_hook(f'layer_{i}_mlp')))
    # Hook any PLE-like submodule if it exists
    for name, submod in layer.named_children():
        if 'embed' in name.lower() or 'ple' in name.lower() or 'per_layer' in name.lower():
            handles.append(submod.register_forward_hook(make_hook(f'layer_{i}_{name}')))

with torch.no_grad():
    out = model(**inputs)

for h in handles:
    h.remove()

print("=== Activation shapes (first 4 layers) ===")
for k, v in sorted(activation_shapes.items()):
    print(f"  {k}: {v}")

In [ ]:
# Final: print summary for copy-paste into architecture.md
print("=== SUMMARY FOR architecture.md ===")
print(f"Model class: {type(model).__name__}")
print(f"Config class: {type(model.config).__name__}")
print(f"Num layers: {len(layers)}")
print(f"Hidden size: {model.config.hidden_size}")
print(f"Num heads: {model.config.num_attention_heads}")
print(f"Num KV heads: {model.config.num_key_value_heads}")
print()
print("PLE state dict keys (copy to architecture.md):")
for k in [k for k in sd.keys() if any(x in k.lower() for x in ['per_layer', 'ple', 'altup', 'lauren'])]:
    print(f"  {k}: {sd[k].shape}")